# Extended Automated Evaluation

Evaluates both the GRADE and RoB2 chatbots using:

| Category | Metrics |
|---|---|
| **RAGAS (semantic)** | Context Recall, Faithfulness, Factual Correctness, Answer Relevancy |
| **N-gram overlap** | ROUGE-1, ROUGE-2, ROUGE-L, BLEU |
| **Semantic similarity** | Cosine similarity (answer vs. reference embeddings) |
| **NLP complexity** | Readability (Flesch, SMOG, Gunning Fog), lexical diversity, syntactic complexity |
| **G-Eval** | LLM-scored Accuracy, Completeness, Clarity (1–5) |

Run this notebook after running `grade_rag.ipynb` and `rob2_rag.ipynb`.

In [ ]:
import os, json, re, warnings
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import numpy as np
import pandas as pd
import textstat
import spacy
import nltk
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity

from langchain_openai import ChatOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from ragas import EvaluationDataset, evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    LLMContextRecall, Faithfulness, FactualCorrectness, AnswerRelevancy
)

from dotenv import load_dotenv
load_dotenv()

nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nlp = spacy.load('en_core_web_sm')

from rag_core import get_embeddings

## 1. Configuration

In [ ]:
EVAL_MODEL = os.getenv("EVAL_MODEL_NAME", "google/gemini-2.5-flash")
MODEL_NAME = os.getenv("MODEL_NAME", "unknown")

eval_llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model=EVAL_MODEL,
    temperature=0.0,
)
embeddings = get_embeddings()

print(f"Chat model: {MODEL_NAME}")
print(f"Eval model: {EVAL_MODEL}")

## 2. Load Datasets and Generate Answers

In [ ]:
# GRADE
grade_df = pd.read_csv("evaluation_details/evaluation_gradebook/full_dataset_gradebook.csv")
grade_sample = grade_df.sample(min(50, len(grade_df)), random_state=42).reset_index(drop=True)

from grade_rag import ask_grade, get_grade_contexts  # requires grade_rag to have been executed

print("Generating GRADE answers...")
grade_sample["chatbot_answer"] = grade_sample["Question"].apply(ask_grade)
grade_sample["retrieved_contexts"] = grade_sample["Question"].apply(
    lambda q: get_grade_contexts(q, k=5)
)
print("Done.")

In [ ]:
# RoB2
rob2_path = "evaluation_details/evaluation_rob2/full_dataset_rob2.csv"
rob2_df = None

if os.path.exists(rob2_path):
    rob2_df = pd.read_csv(rob2_path)
    rob2_sample = rob2_df.sample(min(50, len(rob2_df)), random_state=42).reset_index(drop=True)

    from rob2_rag import ask_rob2, get_rob2_contexts  # requires rob2_rag to have been executed

    print("Generating RoB2 answers...")
    rob2_sample["chatbot_answer"] = rob2_sample["question"].apply(ask_rob2)
    rob2_sample["retrieved_contexts"] = rob2_sample["question"].apply(
        lambda q: get_rob2_contexts(q, k=5)
    )
    print("Done.")
else:
    print("RoB2 dataset not found — skipping RoB2 evaluation.")

## 3. RAGAS Evaluation

In [ ]:
def run_ragas(df, q_col, ref_col, ans_col, ctx_col, eval_llm, label):
    dataset = [
        {
            "user_input": row[q_col],
            "retrieved_contexts": row[ctx_col] if isinstance(row[ctx_col], list) else [row[ctx_col]],
            "response": str(row[ans_col]),
            "reference": str(row[ref_col]),
        }
        for _, row in df.iterrows()
    ]
    evaluation_dataset = EvaluationDataset.from_list(dataset)
    evaluator_llm = LangchainLLMWrapper(eval_llm)

    result = evaluate(
        dataset=evaluation_dataset,
        metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), AnswerRelevancy()],
        llm=evaluator_llm,
    )
    ragas_df = result.to_pandas()
    print(f"\n{label} RAGAS averages:")
    print(ragas_df[["context_recall", "faithfulness", "factual_correctness(mode=f1)", "answer_relevancy"]].mean())
    return ragas_df

model_slug = MODEL_NAME.split("/")[-1]
eval_slug = EVAL_MODEL.split("/")[-1]

grade_ragas = run_ragas(
    grade_sample, "Question", "Answer", "chatbot_answer", "retrieved_contexts",
    eval_llm, "GRADE"
)
grade_ragas.to_csv(f"evaluation_details/evaluation_gradebook/details_ragas_{model_slug}_{eval_slug}_ext.csv", index=False)

In [ ]:
if rob2_df is not None:
    os.makedirs("evaluation_details/evaluation_rob2", exist_ok=True)
    rob2_ragas = run_ragas(
        rob2_sample, "question", "reference_answer", "chatbot_answer", "retrieved_contexts",
        eval_llm, "RoB2"
    )
    rob2_ragas.to_csv(f"evaluation_details/evaluation_rob2/details_ragas_{model_slug}_{eval_slug}.csv", index=False)

## 4. ROUGE and BLEU Scores

In [ ]:
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
smooth = SmoothingFunction().method1

def compute_ngram_metrics(reference: str, hypothesis: str) -> dict:
    scores = scorer.score(reference, hypothesis)
    ref_tokens = reference.lower().split()
    hyp_tokens = hypothesis.lower().split()
    bleu = sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=smooth) if hyp_tokens else 0.0
    return {
        "rouge1_f": scores['rouge1'].fmeasure,
        "rouge2_f": scores['rouge2'].fmeasure,
        "rougeL_f": scores['rougeL'].fmeasure,
        "bleu": bleu,
    }

def add_ngram_metrics(df, ref_col, ans_col):
    metrics = df.apply(lambda r: compute_ngram_metrics(str(r[ref_col]), str(r[ans_col])), axis=1)
    return pd.concat([df, pd.DataFrame(list(metrics))], axis=1)

grade_sample = add_ngram_metrics(grade_sample, "Answer", "chatbot_answer")
print("GRADE n-gram averages:")
print(grade_sample[["rouge1_f", "rouge2_f", "rougeL_f", "bleu"]].mean())

if rob2_df is not None:
    rob2_sample = add_ngram_metrics(rob2_sample, "reference_answer", "chatbot_answer")
    print("\nRoB2 n-gram averages:")
    print(rob2_sample[["rouge1_f", "rouge2_f", "rougeL_f", "bleu"]].mean())

## 5. Semantic Similarity (Answer vs. Reference)

In [ ]:
def compute_semantic_similarity(references: list[str], answers: list[str], emb_model) -> np.ndarray:
    ref_embs = np.array(emb_model.embed_documents(references))
    ans_embs = np.array(emb_model.embed_documents(answers))
    sims = [cosine_similarity([r], [a])[0][0] for r, a in zip(ref_embs, ans_embs)]
    return np.array(sims)

grade_sims = compute_semantic_similarity(
    grade_sample["Answer"].tolist(), grade_sample["chatbot_answer"].tolist(), embeddings
)
grade_sample["semantic_similarity"] = grade_sims
print(f"GRADE semantic similarity: mean={grade_sims.mean():.3f}, std={grade_sims.std():.3f}")

if rob2_df is not None:
    rob2_sims = compute_semantic_similarity(
        rob2_sample["reference_answer"].tolist(), rob2_sample["chatbot_answer"].tolist(), embeddings
    )
    rob2_sample["semantic_similarity"] = rob2_sims
    print(f"RoB2 semantic similarity: mean={rob2_sims.mean():.3f}, std={rob2_sims.std():.3f}")

## 6. NLP Complexity Metrics (from original evaluation)

In [ ]:
def compute_nlp_metrics(text: str) -> dict:
    doc = nlp(text)
    words = [t.text for t in doc if t.is_alpha]
    sentences = list(doc.sents)
    bigrams = list(nltk.bigrams(words))

    sent_texts = [s.text for s in sentences]
    sent_embs = embeddings.embed_documents(sent_texts) if len(sent_texts) > 1 else []
    if len(sent_embs) > 1:
        sims = cosine_similarity(sent_embs)
        avg_sim = (np.sum(sims) - np.trace(sims)) / (sims.shape[0]**2 - sims.shape[0])
        emb_diversity = 1 - avg_sim
    else:
        emb_diversity = 0

    reasoning_kws = {"because", "therefore", "hence", "so", "since", "thus", "if", "then", "reason"}
    depths = [max([len(list(t.ancestors)) for t in s]) for s in sentences] if sentences else [0]

    return {
        "avg_word_length": np.mean([len(w) for w in words]) if words else 0,
        "avg_sentence_length": np.mean([len(s) for s in sentences]) if sentences else 0,
        "flesch_reading_ease": textstat.flesch_reading_ease(text),
        "flesch_kincaid_grade": textstat.flesch_kincaid_grade(text),
        "smog_index": textstat.smog_index(text),
        "gunning_fog": textstat.gunning_fog(text),
        "type_token_ratio": len(set(words)) / len(words) if words else 0,
        "embedding_diversity": emb_diversity,
        "bigram_diversity": len(set(bigrams)) / len(bigrams) if bigrams else 0,
        "reasoning_keyword_ratio": len([w for w in words if w.lower() in reasoning_kws]) / len(words) if words else 0,
        "average_reasoning_steps": sum(1 for w in words if w.lower() in {"if", "then", "so", "thus"}),
        "inference_complexity_score": np.mean(depths),
    }

def add_nlp_metrics(df, ans_col):
    metrics = df[ans_col].apply(compute_nlp_metrics)
    return pd.concat([df, pd.DataFrame(list(metrics))], axis=1)

grade_sample = add_nlp_metrics(grade_sample, "chatbot_answer")
if rob2_df is not None:
    rob2_sample = add_nlp_metrics(rob2_sample, "chatbot_answer")

## 7. G-Eval (LLM-Scored Quality)

In [ ]:
GEVAL_PROMPT = """\
Evaluate the following chatbot answer. Score each dimension 1-5.

Question: {question}
Reference Answer: {reference}
Chatbot Answer: {answer}

Dimensions:
- accuracy: Is the answer factually correct? (1=wrong, 5=fully correct)
- completeness: Does it cover all key points? (1=major gaps, 5=complete)
- clarity: Is it well-written and clear? (1=confusing, 5=excellent)

Think step-by-step, then return ONLY valid JSON:
{{"accuracy": <int>, "completeness": <int>, "clarity": <int>, "reasoning": "<one sentence>"}}
"""

def geval_score(question, reference, answer, llm) -> dict:
    prompt = GEVAL_PROMPT.format(
        question=question,
        reference=str(reference)[:800],
        answer=str(answer)[:800],
    )
    try:
        resp = llm.invoke(prompt)
        content = resp.content if hasattr(resp, 'content') else str(resp)
        content = re.sub(r'^```(?:json)?\n?', '', content.strip())
        content = re.sub(r'\n?```$', '', content.strip())
        return json.loads(content)
    except Exception as e:
        return {"accuracy": None, "completeness": None, "clarity": None, "reasoning": str(e)}

def add_geval(df, q_col, ref_col, ans_col, llm):
    scores = df.apply(
        lambda r: geval_score(r[q_col], r[ref_col], r[ans_col], llm), axis=1
    )
    scores_df = pd.DataFrame(list(scores)).add_prefix("geval_")
    return pd.concat([df, scores_df], axis=1)

print("Running G-Eval on GRADE...")
grade_sample = add_geval(grade_sample, "Question", "Answer", "chatbot_answer", eval_llm)

if rob2_df is not None:
    print("Running G-Eval on RoB2...")
    rob2_sample = add_geval(rob2_sample, "question", "reference_answer", "chatbot_answer", eval_llm)

## 8. Merge RAGAS + All Metrics and Save

In [ ]:
RAGAS_COLS = ["context_recall", "faithfulness", "factual_correctness(mode=f1)", "answer_relevancy"]

def merge_and_save(sample_df, ragas_df, domain, model_slug, eval_slug, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    for col in RAGAS_COLS:
        if col in ragas_df.columns:
            sample_df[col] = ragas_df[col].values

    sample_df.to_csv(f"{out_dir}/full_metrics_{model_slug}_{eval_slug}.csv", index=False)

    numeric_cols = sample_df.select_dtypes(include=np.number).columns.tolist()
    avg_row = sample_df[numeric_cols].mean().to_frame().T
    avg_row.insert(0, "model_name", MODEL_NAME)
    avg_row.insert(1, "domain", domain)

    avg_path = f"{out_dir}/average_metrics.csv"
    if os.path.exists(avg_path):
        avg_row.to_csv(avg_path, mode="a", header=False, index=False)
    else:
        avg_row.to_csv(avg_path, index=False)

    print(f"\n{domain} key averages:")
    key_metrics = [c for c in [
        "context_recall", "faithfulness", "factual_correctness(mode=f1)", "answer_relevancy",
        "rouge1_f", "rougeL_f", "bleu", "semantic_similarity",
        "geval_accuracy", "geval_completeness", "geval_clarity"
    ] if c in sample_df.columns]
    print(sample_df[key_metrics].mean())

merge_and_save(grade_sample, grade_ragas, "GRADE", model_slug, eval_slug,
               "evaluation_details/evaluation_gradebook")

if rob2_df is not None:
    merge_and_save(rob2_sample, rob2_ragas, "RoB2", model_slug, eval_slug,
                   "evaluation_details/evaluation_rob2")